# Experiment: TrustSignal Signal Accuracy

Objective:
- Measure AI signal extraction quality before proof generation.
- Track precision/recall and false-positive rate by vertical and category.
- Tune risk-score thresholds per vertical.


In [ ]:
from __future__ import annotations

import csv
import random
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

SEED = 42
random.seed(SEED)

NOTEBOOK_ROOT = Path.cwd()
DATA_DIR = NOTEBOOK_ROOT / "notebooks" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "signal-eval.csv"

{"seed": SEED, "data_path": str(DATA_PATH)}


## Plan

- Hypothesis: tuned thresholds reduce false positives while preserving recall.
- Variables to sweep: vertical, document category, input type, threshold.
- Metrics: precision, recall, F1, and false positive rate (FPR).


In [ ]:
verticals = ["deed", "credential", "kyc"]
input_types = ["ocr_text", "metadata", "signature_graph", "id_face_match"]
doc_categories = {
    "deed": ["quitclaim", "warranty", "refinance"],
    "credential": ["license", "certificate", "enrollment"],
    "kyc": ["passport", "drivers_license", "utility_bill"],
}

verticals, input_types


## Load Data (or Generate a Reproducible Synthetic Baseline)

Expected CSV columns:
`vertical,doc_category,input_type,y_true,risk_score`


In [ ]:
def generate_synthetic_rows(n_per_vertical: int = 180):
    rows = []
    for v in verticals:
        cats = doc_categories[v]
        for _ in range(n_per_vertical):
            category = random.choice(cats)
            input_type = random.choice(input_types)
            y_true = 1 if random.random() < 0.28 else 0

            # Simulate scoring distributions per class
            if y_true == 1:
                base = random.uniform(0.45, 0.98)
            else:
                base = random.uniform(0.01, 0.72)

            # Vertical-specific noise profiles
            if v == "deed":
                score = min(max(base + random.uniform(-0.04, 0.04), 0.0), 1.0)
            elif v == "credential":
                score = min(max(base + random.uniform(-0.06, 0.06), 0.0), 1.0)
            else:
                score = min(max(base + random.uniform(-0.08, 0.08), 0.0), 1.0)

            rows.append(
                {
                    "vertical": v,
                    "doc_category": category,
                    "input_type": input_type,
                    "y_true": y_true,
                    "risk_score": round(score, 4),
                }
            )
    return rows


def load_rows_from_csv(path: Path):
    with path.open("r", newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        rows = []
        for row in r:
            rows.append(
                {
                    "vertical": row["vertical"],
                    "doc_category": row["doc_category"],
                    "input_type": row["input_type"],
                    "y_true": int(row["y_true"]),
                    "risk_score": float(row["risk_score"]),
                }
            )
        return rows


if DATA_PATH.exists():
    rows = load_rows_from_csv(DATA_PATH)
    source = "csv"
else:
    rows = generate_synthetic_rows()
    source = "synthetic"

{"row_count": len(rows), "source": source}


## Metric Functions


In [ ]:
def confusion_for(rows_subset, threshold: float):
    tp = fp = tn = fn = 0
    for row in rows_subset:
        pred = 1 if row["risk_score"] >= threshold else 0
        truth = row["y_true"]
        if pred == 1 and truth == 1:
            tp += 1
        elif pred == 1 and truth == 0:
            fp += 1
        elif pred == 0 and truth == 0:
            tn += 1
        else:
            fn += 1
    return tp, fp, tn, fn


def metrics(tp, fp, tn, fn):
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "fpr": round(fpr, 4),
        "f1": round(f1, 4),
    }


## Precision/Recall by Vertical and Input Type


In [ ]:
DEFAULT_THRESHOLD = 0.65

grouped = defaultdict(list)
for row in rows:
    grouped[(row["vertical"], row["input_type"])].append(row)

vertical_input_metrics = []
for (vertical, input_type), subset in sorted(grouped.items()):
    tp, fp, tn, fn = confusion_for(subset, DEFAULT_THRESHOLD)
    m = metrics(tp, fp, tn, fn)
    vertical_input_metrics.append(
        {
            "vertical": vertical,
            "input_type": input_type,
            "n": len(subset),
            **m,
        }
    )

vertical_input_metrics[:12]


## False Positive Rate by Document Category


In [ ]:
cat_grouped = defaultdict(list)
for row in rows:
    cat_grouped[(row["vertical"], row["doc_category"])].append(row)

fpr_by_category = []
for (vertical, category), subset in sorted(cat_grouped.items()):
    tp, fp, tn, fn = confusion_for(subset, DEFAULT_THRESHOLD)
    m = metrics(tp, fp, tn, fn)
    fpr_by_category.append(
        {
            "vertical": vertical,
            "doc_category": category,
            "n": len(subset),
            "fpr": m["fpr"],
            "precision": m["precision"],
            "recall": m["recall"],
        }
    )

fpr_by_category


## Threshold Tuning per Vertical


In [ ]:
thresholds = [round(x / 100, 2) for x in range(30, 91, 5)]
FPR_CAP = 0.08

recommendations = {}
threshold_grid = []

for vertical in verticals:
    subset = [r for r in rows if r["vertical"] == vertical]
    scores = []
    for t in thresholds:
        tp, fp, tn, fn = confusion_for(subset, t)
        m = metrics(tp, fp, tn, fn)
        row = {"vertical": vertical, "threshold": t, **m}
        scores.append(row)
        threshold_grid.append(row)

    under_cap = [r for r in scores if r["fpr"] <= FPR_CAP]
    pool = under_cap if under_cap else scores
    best = max(pool, key=lambda r: (r["f1"], r["precision"], r["recall"]))
    recommendations[vertical] = best

{
    "fpr_cap": FPR_CAP,
    "recommended_thresholds": recommendations,
}


## Results


In [ ]:
result = {
    "dataset_source": source,
    "rows": len(rows),
    "default_threshold": DEFAULT_THRESHOLD,
    "recommended_thresholds": {v: rec["threshold"] for v, rec in recommendations.items()},
    "best_f1_by_vertical": {v: rec["f1"] for v, rec in recommendations.items()},
}

result


## Next Steps

- Replace synthetic rows with real sampled document sets for deed, credential, and KYC.
- Split `input_type` results by model version to detect regressions.
- Persist threshold decisions into API risk-scoring config only after a holdout evaluation.
